[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C53_RealTime_Detectors_Course/05_benchmark/05_benchmark.ipynb)

# 05 · 延迟-精度权衡与实时检测器选型（延迟预算 / 端到端模型 / FPS 折算 / 帕累托 / p99）

目标：把「选哪个检测器」从一个凭感觉的问题，变成一个**有硬约束、有可复算数字、有敏感性分析**的问题。

本 notebook 你会亲手实现：
1. **延迟预算分解器**：帧周期 vs 反应延迟，以及「毫秒 → 米」的换算
2. **端到端延迟模型**：预处理 / H2D / 推理 / 解码 / **NMS（依赖目标数）** / D2H / 跟踪
3. **论文 FPS 折算器**：把 108 FPS 一步步折算到车端的真实数字
4. **测量方法学**：warmup 的影响、异步计时的假加速、**p50/p90/p99/p99.9**、热态降频
5. **精度档的总账**：为什么「INT8 掉 1.2 AP」这个说法本身就是错的问法
6. **帕累托前沿筛选**，以及「改一个测量约定，前沿成员就变了」的演示
7. **延迟稳定性**：含 NMS / 无 NMS / top-k 封顶 三种方案的 p50-p99 对比
8. **TSR 选型决策脚本**：硬约束过滤 → 加权打分 → **权重敏感性分析**

> 心智模型：**延迟不是一个数，是一个分布；「谁更快」不是一个关于模型的问题，是一个关于口径的问题。**

## 1 · 延迟预算：33 ms 是帧周期，不是预算

In [ ]:
import numpy as np, math

def frame_period_ms(fps):
    """帧周期：吞吐约束。每 T 毫秒必须消化一帧，否则丢帧或积压。"""
    return 1000.0 / fps

def travel_m(speed_kmh, latency_ms):
    """把毫秒翻译成米 —— 需求评审里唯一需要的换算工具。"""
    return speed_kmh / 3.6 * latency_ms / 1000.0

REACTION = [('相机曝光+读出', 12.0), ('ISP', 8.0), ('传输到 SoC', 2.0),
            ('感知（全部任务）', 45.0), ('融合与跟踪', 8.0),
            ('预测', 10.0), ('规划', 20.0), ('控制+执行器', 40.0)]

tot, acc = sum(v for _, v in REACTION), 0.0
print(f"{'阶段':<18s} {'耗时 ms':>9s} {'累计 ms':>9s} {'累计行驶 m @120km/h':>21s}")
for name, v in REACTION:
    acc += v
    print(f'{name:<18s} {v:>9.1f} {acc:>9.1f} {travel_m(120, acc):>21.2f}')

print(f'\n帧周期 @30 FPS   = {frame_period_ms(30):>6.1f} ms   ← 吞吐约束')
print(f'端到端反应延迟   = {tot:>6.1f} ms   ← 安全约束（**是帧周期的 4.4 倍**）')
print(f'@120 km/h 车已开出 {travel_m(120, tot):.2f} m；每省 10 ms = 少开 {travel_m(120, 10):.2f} m')
assert abs(frame_period_ms(30) - 33.3333) < 1e-3
assert tot == 145.0
assert abs(travel_m(120, tot) - 4.8333) < 1e-3
assert abs(travel_m(120, 10) - 0.3333) < 1e-3
print('\n✅ 吞吐与延迟是两个独立约束：流水线系统可以「30 FPS 吞吐 + 150 ms 延迟」。')

In [ ]:
# 感知的 45 ms 里，TSR 只是任务之一（多数量产系统分时串行，因为要保证各自 WCET 可界定）
PERCEPTION = [('BEV 3D 检测', 18.0), ('占用栅格', 10.0), ('车道线', 5.0),
              ('红绿灯', 4.0), ('TSR', 6.0), ('其他', 2.0)]
p_tot = sum(v for _, v in PERCEPTION)
print(f"{'感知任务':<14s} {'时隙 ms':>9s} {'占感知预算':>11s}")
for n, v in PERCEPTION:
    mark = '   ← 本课的主角' if n == 'TSR' else ''
    print(f'{n:<14s} {v:>9.1f} {v/p_tot:>10.1%}{mark}')
tsr = dict(PERCEPTION)['TSR']
print(f'\n感知总预算 {p_tot:.0f} ms（与反应链里的 45 ms 对齐）')
print(f'**TSR 的真实预算 = {tsr:.0f} ms**，而不是帧周期的 {frame_period_ms(30):.1f} ms —— 差 {frame_period_ms(30)/tsr:.1f} 倍')
assert p_tot == 45.0
assert abs(tsr / p_tot - 0.13333) < 1e-4
assert tsr < frame_period_ms(30) / 5
print('\n⚠️  在没搞清楚「你的预算是分配给你的还是你占用的」之前，')
print('    任何「我的模型跑 8 ms」的陈述都是没有意义的。')

## 2 · 端到端延迟的完整拆解

只有 **NMS** 与 **跟踪** 依赖这一帧里有什么；其余各项在给定模型与分辨率后是常数。

In [ ]:
def pipeline(n_obj, model_ms=4.20, use_nms=True, topk=None,
             h=1080, w=1920, bytes_per=1, bw_gbps=16.0):
    """端到端延迟模型。n_obj = 这一帧里的真实目标数。
       n_cand = 过 score 阈值后进 NMS 的候选框数（随目标数增长，上限 8400 个 anchor）。"""
    n_obj = int(n_obj)
    n_cand = min(300 + 60 * n_obj, 8400)
    if topk is not None:
        n_cand = min(n_cand, topk)                     # ← 硬上限：把 WCET 钉死
    st = {
        '预处理(resize/归一化)': 1.80,
        'H2D 拷贝':             h * w * 3 * bytes_per / (bw_gbps * 1e9) * 1e3,
        '模型推理':             model_ms,
        '解码(anchor/sigmoid)':  0.05 + 2e-5 * n_cand,
        'NMS':                  (3.5e-4 * n_cand + 1.0e-5 * n_cand * n_obj) if use_nms else 0.0,
        'D2H 拷贝':             0.06,
        '跟踪与关联':            0.20 + 0.02 * n_obj,
    }
    return st, sum(st.values()), n_cand

print(f"{'阶段':<22s} {'稀疏场景 n=8':>13s} {'密集路口 n=140':>15s} {'涨幅':>8s}")
s8, t8, c8 = pipeline(8)
s140, t140, c140 = pipeline(140)
for k in s8:
    ratio = s140[k] / s8[k] if s8[k] > 0 else float('inf')
    print(f'{k:<22s} {s8[k]:>13.3f} {s140[k]:>15.3f} {ratio:>7.1f}x')
print(f'{"合计":<22s} {t8:>13.3f} {t140:>15.3f} {t140/t8:>7.1f}x')
print(f'{"候选框数 n_cand":<22s} {c8:>13d} {c140:>15d}')
print(f'\nNMS 占比: 稀疏场景 {s8["NMS"]/t8:.1%}  ->  密集路口 **{s140["NMS"]/t140:.1%}**')
assert t140 > 2.5 * t8
assert s8['NMS'] / t8 < 0.10 and s140['NMS'] / t140 > 0.50
print('✅ 同一个模型、同一块芯片，延迟差 3.4 倍 —— 差别全部来自「这一帧里有什么」。')

In [ ]:
# 拷贝的账：一个能立刻省下毫秒的决定
def copy_ms(h, w, c, bytes_per, bw_gbps=16.0):
    return h * w * c * bytes_per / (bw_gbps * 1e9) * 1e3

u8  = copy_ms(1080, 1920, 3, 1)
f32 = copy_ms(1080, 1920, 3, 4)
print(f'1920x1080x3  uint8 上传: {u8:.3f} ms   ({1080*1920*3:,} B)')
print(f'1920x1080x3  fp32  上传: {f32:.3f} ms   ({1080*1920*3*4:,} B)   <- 贵 {f32/u8:.0f} 倍')
print(f'\n六路相机: uint8 {u8*6:.2f} ms  vs  fp32 {f32*6:.2f} ms   -> 白白多花 {(f32-u8)*6:.2f} ms')
assert abs(u8 - 0.38880) < 1e-5
assert abs(f32 / u8 - 4.0) < 1e-12
assert abs((f32 - u8) * 6 - 6.9984) < 1e-3
print('\n⚠️  `transforms.ToTensor()+Normalize()` 在 CPU 上把图变成 fp32 再 .cuda()')
print('    —— 训练时无所谓（worker 并行预取），**推理时是串行的，直接进 p99**。')
print('✅ 正确做法：只传 uint8，归一化/通道重排/letterbox 都放到设备侧做。')
print('   代价：GPU 的 resize 插值与 OpenCV 未必逐像素一致 -> **先建对拍工具，再搬**（C60 模块 01）。')

## 3 · 论文 FPS 的六个变量：为什么 108 FPS 会变成 14.8 FPS

In [ ]:
VARS = [
    ('① batch size',       'batch=32 报吞吐', 'batch=1',        1.70),
    ('② 含不含 NMS',        '常常不含',        '必须含',          1.20),
    ('③ 含不含预处理/拷贝',  '几乎从不含',      '必须含',          1.35),
    ('④ 精度档',            'TRT FP16(常不写)', 'FP16 或 INT8',   1.00),
    ('⑤ 什么卡',            'T4 / V100',       '车端 SoC',        2.60),
    ('⑥ 输入分辨率',        '640x640',         '1280+ (TSR 必需)', 3.40),
]
print(f"{'变量':<20s} {'论文常用口径':<18s} {'车端真实口径':<16s} {'倍数':>6s}")
compound = 1.0
for name, a, b, f in VARS:
    compound *= f
    print(f'{name:<20s} {a:<18s} {b:<16s} {f:>5.2f}x')
print(f'\n六项复合 = {compound:.2f}x   ← 全部对不上时，论文数字与实测能差 **一个数量级以上**')
assert abs(compound - 24.34536) < 1e-4
assert compound > 20
print('\n⚠️  这里没有一项是「论文造假」。论文数字在它自己的口径下是真的、可复现的。')
print('✅ 正确心态：**论文数字必须先折算到你的口径才能比较**，而不是「论文不可信」。')

In [ ]:
def derate(paper_fps, steps):
    """把论文口径的 FPS 逐步折算到目标系统。steps: [(名字, 'mul'|'add', 值)]"""
    t = 1000.0 / paper_fps
    rows = [('论文口径', '', t)]
    for name, kind, v in steps:
        t = t * v if kind == 'mul' else t + v
        rows.append((name, ('x %.2f' % v) if kind == 'mul' else ('+ %.2f ms' % v), t))
    return rows, t, 1000.0 / t

RTDETR_STEPS = [
    ('T4 -> 车端 SoC（同精度算力约 0.53x）',      'mul', 1.90),
    ('输入 640 -> 1280（TSR 小目标必需）',        'mul', 3.40),
    ('预处理（1280 下 resize/letterbox/归一化）', 'add', 5.40),
    ('H2D + D2H 拷贝',                            'add', 1.60),
    ('跟踪与时序关联',                            'add', 0.80),
]
rows, ms, fps = derate(108, RTDETR_STEPS)
print('RT-DETR-R50：论文 108 FPS (T4 / TRT FP16 / batch=1 / 640 / 只计网络前向)\n')
print(f"{'折算步骤':<42s} {'操作':>10s} {'累计 ms':>10s}")
for name, op, t in rows:
    print(f'{name:<42s} {op:>10s} {t:>10.2f}')
print(f'\n车端端到端 = {ms:.2f} ms = **{fps:.1f} FPS**   （论文 108 FPS 的 1/{108/fps:.1f}）')
assert abs(ms - 67.61481481) < 1e-6
assert abs(fps - 14.7896) < 1e-3
assert 108 / fps > 7.0

In [ ]:
# 第二条链：一个「batch=32 吞吐」口径的 YOLO 数字
YOLO_STEPS = [
    ('batch=32 吞吐 -> batch=1 单帧', 'mul', 1.70),
    ('V100 -> 车端 SoC',              'mul', 2.60),
    ('NMS',                           'add', 1.40),
    ('预处理',                        'add', 1.80),
    ('H2D + D2H 拷贝',                'add', 0.45),
]
rows2, ms2, fps2 = derate(500, YOLO_STEPS)
print('某 YOLO：论文 500 FPS (V100 / FP16 / batch=32 / 只计网络前向)\n')
print(f"{'折算步骤':<34s} {'操作':>10s} {'累计 ms':>10s}")
for name, op, t in rows2:
    print(f'{name:<34s} {op:>10s} {t:>10.2f}')
print(f'\n车端端到端 = {ms2:.2f} ms = **{fps2:.1f} FPS**   （论文 500 FPS 的 1/{500/fps2:.1f}）')
assert abs(ms2 - 12.49) < 1e-9
assert abs(fps2 - 80.064) < 1e-2
print('\n⚠️  两条链的**折算比例不同**（7.3x vs 6.2x），因为六个变量对不同模型的敏感度不同：')
print('    · 分辨率翻倍时，密集预测器的候选框数正比于像素数 -> NMS 涨得比推理还快')
print('    · RT-DETR 的 query 数固定，分辨率只影响 backbone/encoder')
print('    · batch=1 时，深而窄的模型比浅而宽的吃亏更多（kernel launch 开销占比上升）')
print('✅ **所以模型排名会随口径改变，不能照抄论文的相对顺序。**')

## 4 · 怎么正确地测：warmup / 同步 / 分布 / 热态

In [ ]:
def simulate_runs(n, steady=9.30, warm=40, spike_p=0.03, seed=0):
    """模拟一次真实的延迟测量：
       · 前 warm 次包含 autotune / 显存分配 / GPU 升频 -> 显著偏慢
       · 稳态有小噪声
       · spike_p 的概率发生抢占/页错误/热节流 -> 右偏长尾"""
    r = np.random.default_rng(seed)
    lat = np.full(n, steady, dtype=float)
    w = min(warm, n)
    lat[:w] *= np.linspace(3.0, 1.03, w)
    lat += r.normal(0, 0.15, n)
    m = r.random(n) < spike_p
    lat[m] += r.gamma(2.0, 1.8, int(m.sum()))
    return np.maximum(lat, 0.5)

runs = simulate_runs(200, warm=40, seed=1)
mean_all, mean_after = runs.mean(), runs[40:].mean()
LAUNCH_MS = 0.15                                   # kernel launch 本身的耗时

print(f'{"❌ 不丢 warmup 的 200 次均值":<32s} {mean_all:>8.2f} ms  -> {1000/mean_all:>6.1f} FPS')
print(f'{"✅ 丢掉前 40 次的稳态均值":<32s} {mean_after:>8.2f} ms  -> {1000/mean_after:>6.1f} FPS')
print(f'{"❌ 不同步（只测到 kernel launch）":<32s} {LAUNCH_MS:>8.2f} ms  -> {1000/LAUNCH_MS:>6.1f} FPS')
print(f'\n不丢 warmup 高估延迟 {mean_all/mean_after-1:.0%}；不同步「加速」了 {mean_after/LAUNCH_MS:.0f} 倍（全是假的）')
assert mean_all > mean_after * 1.10
assert mean_after / LAUNCH_MS > 40
print('\n✅ 纪律 ①：warmup 后丢弃。   纪律 ②：每次计时前后都 synchronize()（或用 CUDA event）。')

In [ ]:
big = simulate_runs(20000, warm=40, seed=7)[40:]
p50, p90, p99, p999 = np.percentile(big, [50, 90, 99, 99.9])
print(f"{'统计量':<10s} {'延迟 ms':>10s} {'折成 FPS':>10s}")
for lbl, v in [('mean', big.mean()), ('p50', p50), ('p90', p90),
               ('p99', p99), ('p99.9', p999), ('max', big.max())]:
    print(f'{lbl:<10s} {v:>10.2f} {1000/v:>10.1f}')
assert p99 > p50 * 1.12
assert p999 >= p99
assert p50 < big.mean() < p99, '均值被尾巴拉高但又低于尾巴 —— 两头不靠'

BUDGET = 12.0
over = float((big > BUDGET).mean())
print(f'\n预算 {BUDGET:.0f} ms：p50={p50:.2f} 看起来「有 {BUDGET-p50:.1f} ms 余量」，'
      f'但 p99={p99:.2f} **已经超预算**')
print(f'超时帧占比 {over:.2%} -> 30 FPS 下平均每 {1/(over*30):.1f} 秒一次')
assert 0.005 < over < 0.06
print('\n✅ 纪律 ③：报分布不报均值。车端安全关心 p99（甚至 p99.9）。')
print('⚠️  但 **p99 也不是上界** —— 它之上还有 p99.9 和真正的 max。')
print('    实时系统真正要的是 WCET：要么删掉数据依赖项，要么给它封顶。')

In [ ]:
def with_thermal(lat, drift_start=0.35, max_slow=1.18):
    """持续满负载后 SoC 降频：延迟缓慢漂移。短测试完全看不出来。"""
    n = len(lat); k = np.ones(n); i0 = int(n * drift_start)
    k[i0:] = np.linspace(1.0, max_slow, n - i0)
    return lat * k

long_run = with_thermal(simulate_runs(20000, warm=40, seed=3))[40:]
early, late = long_run[:2000], long_run[-2000:]
e50, e99 = np.percentile(early, [50, 99])
l50, l99 = np.percentile(late,  [50, 99])
print(f"{'':<22s} {'p50':>8s} {'p99':>8s}")
print(f'{"开测前 2000 帧（冷）":<22s} {e50:>8.2f} {e99:>8.2f}')
print(f'{"持续负载后 2000 帧（热）":<22s} {l50:>8.2f} {l99:>8.2f}')
print(f'\n热态 p50 比冷态高 {l50/e50-1:.0%} —— 「实验室 8 ms，车上跑半小时后 10 ms」就是这么来的')
assert l50 > e50 * 1.10
assert l99 > e99
print('\n✅ 纪律 ④：测够长并覆盖热态。   纪律 ⑤：用真实帧序列，不要同一张图跑 1000 次')
print('   （同一张图 cache 命中异常高，而且完全测不出 NMS 的场景依赖）。')

## 5 · 精度档的总账：「INT8 掉多少点」是个错的问法

In [ ]:
BASE_MS_FP32, BASE_AP = 12.00, 53.10          # 某模型的 FP32 端到端延迟与 AP
PRECISION = [
    ('FP32',       1.00,  0.00, '基线'),
    ('FP16',       0.42, -0.05, 'Tensor Core，几乎白送；只需查 inf/nan'),
    ('INT8 (PTQ)', 0.27, -1.20, '需要**有代表性的**校准集'),
    ('INT8 (QAT)', 0.27, -0.40, '要重训，几个 GPU-天'),
    ('混合精度',    0.34, -0.50, '敏感层保 FP16；分区变多可能反而慢，要实测'),
]
print(f"{'精度档':<12s} {'相对延迟':>8s} {'延迟 ms':>9s} {'AP':>7s} {'说明'}")
for name, mul, dap, note in PRECISION:
    print(f'{name:<12s} {mul:>8.2f} {BASE_MS_FP32*mul:>9.2f} {BASE_AP+dap:>7.2f}  {note}')
fp16_ms = BASE_MS_FP32 * 0.42
int8_ms = BASE_MS_FP32 * 0.27
assert fp16_ms < BASE_MS_FP32 * 0.50
assert int8_ms < fp16_ms * 0.70
print(f'\n✅ FP16 基本白送：延迟降到 {0.42:.0%}，AP 几乎不动，不需要任何数据。')
print('   唯一要做的验证：用极端输入（过曝、超大幅值）跑一遍查 inf/nan')
print('   —— 这个问题在训练时被 loss scaling 掩盖，只在纯 FP16 推理时暴露。')

In [ ]:
# 关键实验：**在同一延迟预算下**，INT8 的大模型 vs FP16 的小模型，谁的 AP 高？
FAMILY_FP16 = [('RT-DETR-R18', 46.5, 4.6), ('RT-DETR-R34', 48.9, 6.2),
               ('RT-DETR-R50', 53.1, 9.3), ('RT-DETR-R101', 54.3, 13.5)]
FAMILY = [(n, ap, ms / 0.42) for n, ap, ms in FAMILY_FP16]      # 反推 FP32 端到端
BUDGET_MS, INT8_AP_COST = 6.0, 1.20

def best_in_budget(family, mul, ap_cost, budget):
    ok = [(ap - ap_cost, n, ms * mul) for n, ap, ms in family if ms * mul <= budget]
    if not ok:
        return None
    ap, n, ms = max(ok)
    return n, ap, ms

bf16 = best_in_budget(FAMILY, 0.42, 0.00, BUDGET_MS)
bi8  = best_in_budget(FAMILY, 0.27, INT8_AP_COST, BUDGET_MS)
print(f'延迟预算 {BUDGET_MS:.1f} ms 下能选的最大模型：\n')
print(f"{'精度档':<12s} {'能上的最大模型':<16s} {'实际延迟 ms':>12s} {'实际 AP':>9s}")
print(f'{"FP16":<12s} {bf16[0]:<16s} {bf16[2]:>12.2f} {bf16[1]:>9.2f}')
print(f'{"INT8 (PTQ)":<12s} {bi8[0]:<16s} {bi8[2]:>12.2f} {bi8[1]:>9.2f}')
assert bf16[0] == 'RT-DETR-R18' and bi8[0] == 'RT-DETR-R50'
assert bi8[1] > bf16[1] + 4.0
print(f'\n**INT8 本身掉 {INT8_AP_COST:.1f} AP，但让你在同一预算下从 {bf16[1]:.1f} AP 提到 {bi8[1]:.1f} AP（+{bi8[1]-bf16[1]:.1f}）。**')
print('✅ 所以正确的问法不是「INT8 掉多少点」，而是')
print('   「在同一延迟预算下，INT8 的大模型 vs FP16 的小模型，谁的 AP 更高」。')
print('⚠️  前提是校准集必须是**线上分布的有代表性样本**（含夜间/逆光/雨雾/隧道口/长尾类别），')
print('    用训练集前 500 张（很可能全是晴天白天）会得到只对晴天正确的量化范围。')

## 6 · 帕累托前沿：被支配的模型直接删掉

下表是**教学用近似值**（口径：batch=1 / TRT FP16 / 640 / 含预处理）。
`needs_nms=True` 的模型要额外加上 NMS 时间 —— 这个「加多少」正是要演示的口径变量。

In [ ]:
MODELS = [   # (名字, COCO AP, 不含 NMS 的端到端 ms, 是否需要 NMS)
    ('RTMDet-S',     44.5,  2.6, True),
    ('RTMDet-M',     49.1,  4.4, True),
    ('RTMDet-L',     51.3,  6.1, True),
    ('YOLOv8-S',     44.9,  4.0, True),
    ('YOLOv8-M',     50.2,  6.5, True),
    ('YOLOv8-L',     52.9,  9.5, True),
    ('YOLOv8-X',     53.9, 15.0, True),
    ('RT-DETR-R18',  46.5,  4.6, False),
    ('RT-DETR-R34',  48.9,  6.2, False),
    ('RT-DETR-R50',  53.1,  9.3, False),
    ('RT-DETR-R101', 54.3, 13.5, False),
    ('D-FINE-S',     48.5,  4.8, False),
    ('D-FINE-M',     52.3,  7.1, False),
    ('D-FINE-L',     54.0,  9.8, False),
]

def end_to_end(models, nms_ms):
    return [(n, ap, round(base + (nms_ms if need else 0.0), 3)) for n, ap, base, need in models]

def pareto_front(pts):
    """i 被支配 <=> 存在 j: lat_j <= lat_i 且 ap_j >= ap_i，且至少一项严格更好。"""
    out = []
    for i, (n, a, l) in enumerate(pts):
        dominated = any(l2 <= l and a2 >= a and (l2 < l or a2 > a)
                        for j, (_, a2, l2) in enumerate(pts) if j != i)
        if not dominated:
            out.append((n, a, l))
    return sorted(out, key=lambda r: r[2])

ptsA = end_to_end(MODELS, 1.4)
frontA = pareto_front(ptsA)
nameA = [n for n, _, _ in frontA]
print('口径 A：NMS 计 1.4 ms（朴素实现，密集场景更慢）\n')
print(f"{'模型':<14s} {'AP':>6s} {'端到端 ms':>10s} {'前沿?':>7s}")
for n, a, l in sorted(ptsA, key=lambda r: r[2]):
    print(f'{n:<14s} {a:>6.1f} {l:>10.2f} {("● 是" if n in nameA else "○ 被支配"):>7s}')
print(f'\n前沿成员 {len(frontA)} 个: {nameA}')
assert len(frontA) == 8, frontA
assert 'RT-DETR-R50' in nameA and 'D-FINE-L' in nameA and 'RT-DETR-R101' in nameA
assert 'YOLOv8-L' not in nameA and 'YOLOv8-S' not in nameA and 'RTMDet-L' not in nameA
print('✅ 6 个模型被支配 —— 存在一个「不更慢且不更差」的替代品，可以直接从候选集删掉。')

In [ ]:
# **前沿是口径的函数**：只把 NMS 的计入方式改成 0.5 ms（EfficientNMS plugin + top-k 截断）
ptsB = end_to_end(MODELS, 0.5)
frontB = pareto_front(ptsB)
nameB = [n for n, _, _ in frontB]
print('口径 B：NMS 计 0.5 ms\n')
print(f'前沿成员 {len(frontB)} 个: {nameB}\n')
entered = [n for n in nameB if n not in nameA]
left    = [n for n in nameA if n not in nameB]
print(f'口径 A -> B **新进入前沿**: {entered}')
print(f'口径 A -> B **退出前沿**  : {left}')
assert len(frontB) == 10, frontB
assert set(entered) == {'YOLOv8-S', 'RTMDet-L'}, entered
assert left == []
print('\n⚠️  **同一批模型、同一批 AP，只改一个测量约定，前沿成员就从 8 个变成 10 个。**')
print('✅ 所以「谁在前沿上」这个结论必须和口径一起被引用，否则没有意义。')

In [ ]:
def best_under_budget(pts, budget_ms):
    """选型的核心动作：**同延迟比 AP**，而不是同 AP 比延迟。"""
    ok = [(a, n, l) for n, a, l in pts if l <= budget_ms]
    if not ok:
        return None
    a, n, l = max(ok)
    return n, a, l

print(f"{'延迟预算 ms':>12s} {'最优选择':<16s} {'AP':>7s} {'实际延迟':>9s}")
for bud in [3.0, 5.0, 6.5, 8.0, 10.0, 14.0]:
    r = best_under_budget(ptsA, bud)
    if r is None:
        print(f'{bud:>12.1f} {"无可行方案":<16s} {"—":>7s} {"—":>9s}')
    else:
        print(f'{bud:>12.1f} {r[0]:<16s} {r[1]:>7.1f} {r[2]:>9.2f}')
assert best_under_budget(ptsA, 3.0) is None
assert best_under_budget(ptsA, 5.0)[0] == 'D-FINE-S'
assert best_under_budget(ptsA, 8.0)[0] == 'D-FINE-M'
assert best_under_budget(ptsA, 10.0)[0] == 'D-FINE-L'
print('\n✅ 「达到 53 AP 谁最快」是假问题 —— 你不需要恰好 53 AP，')
print('   你需要「在预算内 AP 最高」。延迟是系统给的硬约束，AP 是要最大化的目标。')

## 7 · 延迟稳定性：p99 才是决定能不能上车的那个数

用第 2 节的端到端模型，跑一段「真实路测」的目标数分布（重尾：多数帧目标少，少数路口帧目标极多）。

In [ ]:
rng = np.random.default_rng(7)
N_SCENE = 4000
n_obj_s = np.clip(rng.lognormal(np.log(8), 0.9, N_SCENE).astype(int) + 1, 1, 220)
print(f'场景目标数分布: p50={np.percentile(n_obj_s,50):.0f}  p90={np.percentile(n_obj_s,90):.0f}  '
      f'p99={np.percentile(n_obj_s,99):.0f}  max={n_obj_s.max()}')

t_yolo = np.array([pipeline(n, model_ms=4.20, use_nms=True)[1]  for n in n_obj_s])
t_detr = np.array([pipeline(n, model_ms=6.30, use_nms=False)[1] for n in n_obj_s])
t_cap  = np.array([pipeline(n, model_ms=4.20, use_nms=True, topk=1000)[1] for n in n_obj_s])

def q(t):
    a, b, c = np.percentile(t, [50, 90, 99])
    return a, b, c, c - a

print(f"\n{'方案':<24s} {'p50':>8s} {'p90':>8s} {'p99':>8s} {'p99-p50':>9s}")
for lbl, t in [('YOLO 式（含 NMS）', t_yolo), ('RT-DETR 式（无 NMS）', t_detr),
               ('YOLO + top-k 封顶(1000)', t_cap)]:
    a, b, c, d = q(t)
    print(f'{lbl:<24s} {a:>8.2f} {b:>8.2f} {c:>8.2f} {d:>9.2f}')

y50, _, y99, ygap = q(t_yolo)
d50, _, d99, dgap = q(t_detr)
c50, _, c99, cgap = q(t_cap)
assert y50 < d50,  'YOLO 的中位延迟更低'
assert y99 > d99,  '但 YOLO 的 p99 反超'
assert ygap > 3 * dgap
assert c99 < y99 - 2.0
print(f'\n⚠️  **按 p50 选 -> YOLO；按 p99 选 -> RT-DETR。同一份数据，两个相反的结论。**')
print(f'    而「按均值选」会给出和「按 p50 选」一样的答案，然后上车在城市路口掉帧。')
print(f'\n注意 RT-DETR 式的 p99-p50 也不是 0（{dgap:.2f} ms）—— 那是**跟踪与关联**的目标数依赖。')
print('    所以严格说法是「删掉了方差最大的那一项」，不是「延迟完全恒定」。')

In [ ]:
# top-k 封顶的代价：哪些帧被截断，截掉了多少候选
n_cand_full = np.minimum(300 + 60 * n_obj_s, 8400)
truncated = n_cand_full > 1000
frac = float(truncated.mean())
dropped = np.where(truncated, (n_cand_full - 1000) / n_cand_full, 0.0)
print(f'候选框数超过上限 1000 的帧占比: **{frac:.1%}**')
print(f'这些帧平均被截掉 {dropped[truncated].mean():.1%} 的候选（全部是低分候选）')
print(f'最坏的一帧被截掉 {dropped.max():.1%}')
assert 0.20 < frac < 0.45
assert dropped[truncated].mean() > 0.15
print('\n⚠️  被截掉的低分候选，主要就是**远处的小目标**与**部分遮挡的目标**')
print('    —— 这个优化恰好在最需要召回的场景里牺牲召回。TSR 里尤其危险：')
print('    一块 80 m 外刚进视野的限速牌分数天然低，在拥挤路口很可能就是被截掉的那个。')
print('\n✅ 封顶必须配三件事：')
print('   ① 统计「候选数 > N_max」的帧占比（太高说明 N_max 设小了）')
print('   ② 建一个「密集路口」评测切片，对比封顶前后的召回')
print('   ③ 把截断事件打点上报，作为线上监控指标')

## 8 · TSR 选型决策脚本：硬约束过滤 → 加权打分 → 敏感性分析

关键设计：**硬约束过滤和加权打分严格分开**。先打分再看约束，会得到一个分数很高但根本上不了车的方案。

In [ ]:
CANDIDATES = [
    dict(name='YOLOv8-M',    ap_small=28.4, p99=12.6, mem=740,  plugin=False, menu=False, tool=0.95),
    dict(name='RTMDet-M',    ap_small=29.1, p99=11.8, mem=690,  plugin=False, menu=False, tool=0.85),
    dict(name='RT-DETR-R50', ap_small=34.8, p99=11.9, mem=1180, plugin=True,  menu=True,  tool=0.70),
    dict(name='RT-DETRv2-S', ap_small=31.6, p99= 8.4, mem=820,  plugin=False, menu=True,  tool=0.70),
    dict(name='D-FINE-M',    ap_small=33.2, p99= 9.9, mem=960,  plugin=True,  menu=True,  tool=0.50),
    dict(name='YOLOv8-X',    ap_small=31.9, p99=19.4, mem=1420, plugin=False, menu=False, tool=0.95),
]
# ap_small = 在**自己数据上按像素尺寸分桶**得到的小目标 AP（不是 COCO 总 AP！）
# menu     = 是否原生支持「一份权重多档速度」   tool = 工具链成熟度 0~1

HARD_STRICT = dict(p99_max=12.0, mem_max=1024, allow_plugin=False)
HARD_RELAX  = dict(p99_max=12.0, mem_max=1280, allow_plugin=True)
W_BALANCED  = dict(ap=0.45, head=0.20, menu=0.20, tool=0.15)
W_SMALLOBJ  = dict(ap=0.70, head=0.10, menu=0.10, tool=0.10)

def reject_reasons(c, hard):
    r = []
    if c['p99'] > hard['p99_max']:
        r.append('p99 %.1f > %.1f ms' % (c['p99'], hard['p99_max']))
    if c['mem'] > hard['mem_max']:
        r.append('显存 %d > %d MB' % (c['mem'], hard['mem_max']))
    if c['plugin'] and not hard['allow_plugin']:
        r.append('需要 TRT plugin')
    return r

def score(c, hard, w):
    return (w['ap']   * (c['ap_small'] / 40.0)
          + w['head'] * ((hard['p99_max'] - c['p99']) / hard['p99_max'])
          + w['menu'] * (1.0 if c['menu'] else 0.0)
          + w['tool'] * c['tool'])

def select_detector(cands, hard, w):
    ranked, rejected = [], []
    for c in cands:
        r = reject_reasons(c, hard)
        if r:
            rejected.append((c['name'], r))
        else:
            ranked.append((score(c, hard, w), c['name']))
    ranked.sort(reverse=True)
    return ranked, rejected

def show(tag, ranked, rejected):
    print(f'—— {tag} ——')
    for s, n in ranked:
        print(f'   {n:<14s} 得分 {s:.4f}')
    for n, r in rejected:
        print(f'   {n:<14s} ❌ 淘汰: {"; ".join(r)}')
    print()

r1, x1 = select_detector(CANDIDATES, HARD_STRICT, W_BALANCED)
show('S1 保守：p99<=12ms / 显存<=1024MB / 不允许 plugin，均衡权重', r1, x1)
assert r1[0][1] == 'RT-DETRv2-S', r1
assert len(x1) == 4 and len(r1) == 2
print('读法：**4 个候选在硬约束这一步就被淘汰了，根本轮不到打分。**')

In [ ]:
r2, x2 = select_detector(CANDIDATES, HARD_RELAX, W_BALANCED)
show('S2 放开 plugin 与显存，权重不变', r2, x2)
assert r2[0][1] == 'RT-DETRv2-S', r2
assert len(x2) == 2 and len(r2) == 4
print('读法：放开两条约束后候选从 2 个变 4 个，但**胜出者没变**')
print('      -> 说明 plugin 与显存不是这次决策的瓶颈，不用在这两件事上开会。\n')

r3, x3 = select_detector(CANDIDATES, HARD_RELAX, W_SMALLOBJ)
show('S3 同 S2 的约束，但把小目标 AP 的权重从 0.45 提到 0.70', r3, x3)
assert r3[0][1] == 'RT-DETR-R50', r3
assert r2[0][1] != r3[0][1]
print('读法：**只改权重，胜出者就换了。**')
print('      -> 真正需要开会讨论的不是「选哪个模型」，而是「小目标 AP 到底值多少权重」。')
print('      -> 而这个问题只能用数据回答：统计各尺寸桶的目标占比 + 各桶漏检的安全后果。')
print('\n✅ 选型脚本的价值不是给出答案，是**把假设摆到台面上并做敏感性分析**。')

## ✏️ 练习 1：毫秒 → 米

实现两个互逆的函数：
- `travel_distance_m(speed_kmh, latency_ms)`：这段延迟里车开出多少米
- `latency_budget_ms(speed_kmh, meters)`：要把「多开的距离」控制在 `meters` 以内，延迟上限是多少毫秒

In [ ]:
def travel_distance_m(speed_kmh, latency_ms):
    # TODO
    raise NotImplementedError

def latency_budget_ms(speed_kmh, meters):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测（手算）——
assert abs(travel_distance_m(120, 150) - 5.0) < 1e-12          # 33.3333 m/s * 0.15 s
assert abs(travel_distance_m(60, 100) - 60/3.6*0.1) < 1e-12
assert abs(travel_distance_m(120, 0) - 0.0) < 1e-12
assert abs(latency_budget_ms(120, 1.0) - 30.0) < 1e-9          # 1.0 m / 33.3333 m/s = 30 ms
assert abs(latency_budget_ms(36, 0.5) - 50.0) < 1e-9           # 36 km/h = 10 m/s
for v, t in [(120, 145.0), (80, 33.3), (36, 6.0)]:             # 互逆性
    assert abs(latency_budget_ms(v, travel_distance_m(v, t)) - t) < 1e-9
print(f"{'车速 km/h':>10s} {'145 ms 开出 m':>14s} {'控制在 1 m 内的延迟上限 ms':>26s}")
for v in [36, 60, 80, 120, 150]:
    print(f'{v:>10d} {travel_distance_m(v, 145):>14.2f} {latency_budget_ms(v, 1.0):>26.1f}')
print('\n✅ 练习 1 通过：把毫秒翻译成米，是判断「这个优化值不值得做」的唯一工具。')

## ✏️ 练习 2：论文 FPS 折算器

实现 `derate_chain(paper_fps, steps)` 返回 `(final_ms, real_fps)`。
`steps` 是 `[(名字, 'mul'|'add', 值)]`，按顺序依次应用：`'mul'` 乘在当前毫秒数上，`'add'` 加上去。

In [ ]:
def derate_chain(paper_fps, steps):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测（手算）——
ms_, fps_ = derate_chain(100, [('a', 'mul', 1.5), ('b', 'mul', 2.0), ('c', 'add', 2.0)])
assert abs(ms_ - 32.0) < 1e-9, ms_          # 10 -> 15 -> 30 -> 32
assert abs(fps_ - 31.25) < 1e-9, fps_
ms0, fps0 = derate_chain(50, [])
assert abs(ms0 - 20.0) < 1e-12 and abs(fps0 - 50.0) < 1e-12
ms1, fps1 = derate_chain(108, RTDETR_STEPS)
assert abs(ms1 - 67.61481481) < 1e-6 and abs(fps1 - 14.7896) < 1e-3
ms2, fps2 = derate_chain(500, YOLO_STEPS)
assert abs(ms2 - 12.49) < 1e-9 and abs(fps2 - 80.064) < 1e-2
print(f'RT-DETR-R50 : 论文 108 FPS -> 车端 {fps1:5.1f} FPS  ({ms1:.2f} ms)')
print(f'某 YOLO     : 论文 500 FPS -> 车端 {fps2:5.1f} FPS  ({ms2:.2f} ms)')
print('\n✅ 练习 2 通过：报延迟必须报口径，否则「谁更快」这个问题没有答案。')

## ✏️ 练习 3：帕累托前沿

实现 `pareto(pts)`：`pts` 是 `[(名字, AP, 延迟ms)]`，返回**未被支配**的点，按延迟升序。

支配定义：`i` 被支配 ⟺ 存在 `j != i` 使得 `lat_j <= lat_i` 且 `ap_j >= ap_i`，且至少一项严格更好。

In [ ]:
def pareto(pts):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测（手算）——
toy = [('A', 40, 5.0), ('B', 45, 5.0), ('C', 44, 7.0), ('D', 50, 9.0)]
f = pareto(toy)
assert [n for n, _, _ in f] == ['B', 'D'], f
#  A 被 B 支配（同延迟但 AP 更低）；C 被 B 支配（更慢且 AP 更低）
assert pareto([('X', 50, 5.0)]) == [('X', 50, 5.0)]
# 在真实表上复现主结论
fa = [n for n, _, _ in pareto(end_to_end(MODELS, 1.4))]
fb = [n for n, _, _ in pareto(end_to_end(MODELS, 0.5))]
assert len(fa) == 8 and len(fb) == 10
assert 'YOLOv8-L' not in fa and 'RT-DETR-R50' in fa
assert set(fb) - set(fa) == {'YOLOv8-S', 'RTMDet-L'}
print(f'口径 A (NMS 1.4 ms) 前沿 {len(fa)} 个: {fa}')
print(f'口径 B (NMS 0.5 ms) 前沿 {len(fb)} 个: {fb}')
print('\n✅ 练习 3 通过：**前沿是口径的函数，不是模型的固有属性。**')

## ✏️ 练习 4：TSR 选型决策脚本

实现 `select_tsr(cands, hard, w)` 返回 `(ranked, rejected)`：
- `ranked`：通过全部硬约束的候选，`[(得分, 名字)]`，按得分降序
- `rejected`：`[(名字, [淘汰理由, ...])]`

直接复用上面已定义的 `reject_reasons(c, hard)` 与 `score(c, hard, w)`。

In [ ]:
def select_tsr(cands, hard, w):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
ra, xa = select_tsr(CANDIDATES, HARD_STRICT, W_BALANCED)
assert ra[0][1] == 'RT-DETRv2-S' and len(ra) == 2 and len(xa) == 4
assert abs(ra[0][0] - 0.7205) < 1e-6, ra[0]     # 0.45*0.79 + 0.20*0.30 + 0.20 + 0.15*0.70
rb, xb = select_tsr(CANDIDATES, HARD_RELAX, W_BALANCED)
assert rb[0][1] == 'RT-DETRv2-S' and len(rb) == 4 and len(xb) == 2
rc, xc = select_tsr(CANDIDATES, HARD_RELAX, W_SMALLOBJ)
assert rc[0][1] == 'RT-DETR-R50', rc
assert [s for s, _ in ra] == sorted([s for s, _ in ra], reverse=True), '必须按得分降序'
assert dict(xa)['RT-DETR-R50'], 'R50 在严格约束下应被淘汰并给出理由'
print(f"{'场景':<28s} {'胜出':<16s} {'候选数':>7s} {'被淘汰':>7s}")
for tag, (r, x) in [('S1 严格 / 均衡权重', (ra, xa)),
                    ('S2 放开约束 / 均衡权重', (rb, xb)),
                    ('S3 放开约束 / 重小目标', (rc, xc))]:
    print(f'{tag:<28s} {r[0][1]:<16s} {len(r):>7d} {len(x):>7d}')
print('\n✅ 练习 4 通过：硬约束过滤与加权打分必须分开，且**必须做权重敏感性分析**。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def travel_distance_m(speed_kmh, latency_ms):
    return speed_kmh / 3.6 * latency_ms / 1000.0

def latency_budget_ms(speed_kmh, meters):
    return meters / (speed_kmh / 3.6) * 1000.0

In [ ]:
# 练习 2 参考答案
def derate_chain(paper_fps, steps):
    t = 1000.0 / paper_fps
    for _, kind, v in steps:
        t = t * v if kind == 'mul' else t + v
    return t, 1000.0 / t

In [ ]:
# 练习 3 参考答案
def pareto(pts):
    out = []
    for i, (n, a, l) in enumerate(pts):
        dominated = any(l2 <= l and a2 >= a and (l2 < l or a2 > a)
                        for j, (_, a2, l2) in enumerate(pts) if j != i)
        if not dominated:
            out.append((n, a, l))
    return sorted(out, key=lambda r: r[2])

In [ ]:
# 练习 4 参考答案
def select_tsr(cands, hard, w):
    ranked, rejected = [], []
    for c in cands:
        r = reject_reasons(c, hard)
        if r:
            rejected.append((c['name'], r))
        else:
            ranked.append((score(c, hard, w), c['name']))
    ranked.sort(reverse=True)
    return ranked, rejected

---
## 🧪 真实工程胶囊：一份可直接照抄的测速与选型协议

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 实时检测器测速与选型协议（把它写进团队 wiki，每次选型照着走）
# ══════════════════════════════════════════════════════════════════════

# ─── 0. 先写死口径（Benchmark Contract），所有模型必须用同一份 ───────────
BENCH_CONTRACT = dict(
    device        = "Orin-X / driver 535.x / TensorRT 10.x",   # 换任何一项都要重测
    precision     = "fp16",              # 或 int8 + 校准集哈希
    batch         = 1,                   # 车端就是 1，不要用吞吐口径
    input_size    = (1280, 1280),        # TSR 的真实分辨率，不是 COCO 的 640
    includes      = ["preprocess", "H2D", "infer", "decode", "NMS", "D2H", "track"],
    clock_locked  = True,                # nvidia-smi -lgc / jetson_clocks
    warmup_iters  = 200,                 # 丢弃
    measure_iters = 2000,
    frames        = "real_route_mix_v3", # **真实帧序列**，含稀疏高速/密集路口/夜间
    report        = ["p50", "p90", "p99", "p99.9", "max", "peak_mem_MB"],
)
# 铁律：报 p50 和 p99 **两个数**。只报一个的对比无效。

# ─── 1. 用 trtexec 做交叉验证（它已经把 warmup/同步/分位数做对了）─────────
#   trtexec --loadEngine=m.engine --iterations=2000 --avgRuns=1 #           --percentile=99 --useCudaGraph --dumpProfile --separateProfileRun
#   看三件事：① Latency 的 median/percentile ② 逐层耗时找瓶颈
#             ③ 有没有 "unsupported"/多子图（静默回退 -> 转了但没变快）

# ─── 2. Python 侧的正确计时骨架 ─────────────────────────────────────────
#   for _ in range(200): model(next(frames))          # warmup 丢弃
#   torch.cuda.synchronize()
#   ts = []
#   for _ in range(2000):
#       x = next(frames)                              # 真实帧，不是同一张
#       torch.cuda.synchronize(); t0 = time.perf_counter()
#       out = model(x)
#       torch.cuda.synchronize()                      # ← 少了这句测的是 kernel launch
#       ts.append((time.perf_counter() - t0) * 1e3)
#   p50, p90, p99, p999 = np.percentile(ts, [50, 90, 99, 99.9])

# ─── 3. 热态验证（短测试测不出降频）────────────────────────────────────
#   连续满负载跑 >= 10 分钟，对比「前 2000 帧」与「后 2000 帧」的 p50
#   通过条件：漂移 < 10%；超过就要在预算里预留降频余量

# ─── 4. 选型三步（顺序不能反）──────────────────────────────────────────
HARD_CONSTRAINTS = dict(          # ① 硬约束过滤：多数候选在这一步就没了
    p99_ms_max   = 6.0,           # 你的时隙，不是帧周期
    max_ms_max   = 9.0,           # 最坏值也要有门槛
    peak_mem_MB  = 1024,          # SoC 显存由所有感知任务共享
    allow_plugin = False,         # 每个 TRT plugin 都是长期维护负债
    ptq_ap_drop_max = 2.0,        # PTQ 掉点超过这个就要重新评估
)
METRIC = "AP_small(<32px) on our_val_v7"   # ② **不是 COCO AP**，按像素尺寸分桶
WEIGHTS = dict(ap=0.45, headroom=0.20, speed_menu=0.20, toolchain=0.15)  # ③ 打分
# ④ **必须做敏感性分析**：把主要权重 ±50% 各跑一次。
#    结论变了 -> 真正要定的是那个权重，而不是模型。

# ─── 5. 发布前的验收清单 ───────────────────────────────────────────────
#   [ ] 端到端 p99 <= 时隙，max <= 1.5 x 时隙
#   [ ] 热态 10 分钟后 p50 漂移 < 10%
#   [ ] 峰值显存 <= 上限（与其他感知任务并发时实测，不是单跑）
#   [ ] TRT 子图数 == 1（无静默回退）
#   [ ] 分尺寸桶 AP 全部不低于 baseline（回归门禁）
#   [ ] 密集路口切片的召回不低于 baseline（如果上了 top-k 封顶，这条是硬门槛）
#   [ ] 记录 engine 的构建环境（GPU 型号 + 驱动 + TRT 版本 + 精度 + 校准集哈希）
'''
print(RECIPE)
for token in ['BENCH_CONTRACT', 'p99', 'trtexec', 'synchronize', 'clock_locked',
              'HARD_CONSTRAINTS', 'AP_small', '敏感性分析', '子图数']:
    assert token in RECIPE, token
print('✅ 协议覆盖：口径契约 / trtexec 交叉验证 / 计时骨架 / 热态 / 选型三步 / 验收清单')

### 小结

- **33 ms 是帧周期，不是延迟预算。** 吞吐与延迟是两个独立约束。端到端反应链约 145 ms，
  感知只占 45 ms，而 TSR 在共享 SoC 上只分到约 **6 ms**。
  换算工具只有一个：**毫秒 → 米**（120 km/h 下每 10 ms = 0.33 m）。
- **端到端延迟 = 预处理 + H2D + 推理 + 解码 + NMS + D2H + 跟踪。**
  只有 NMS 与跟踪依赖场景；同一个模型在稀疏场景与密集路口能差 3.4 倍。
  一个立刻能拿的收益：**只上传 uint8，归一化放设备侧**（fp32 上传贵 4 倍）。
- **论文 FPS 不可信的六个变量**：batch / 含不含 NMS / 含不含预处理与拷贝 / 精度档 / 什么卡 /
  输入分辨率。复合起来能差 **24 倍**。108 FPS 折到车端 TSR 场景是 **14.8 FPS**。
  而且**六个变量对不同模型的敏感度不同 → 排名会随口径改变**，不能照抄论文的相对顺序。
- **五条测量纪律**：warmup 后丢弃 / 每次计时都同步 / 报分布不报均值 / 测够长覆盖热态 /
  用真实帧序列。不同步能测出 **60 倍的假加速**；不丢 warmup 高估约 **20%**。
- **精度档的正确问法**不是「INT8 掉多少点」，而是「同一预算下 INT8 的大模型 vs FP16 的小模型
  谁 AP 高」——本例里 INT8 掉 1.2 AP，却让同预算下的 AP 从 46.5 涨到 51.9。
  前提是**校准集必须覆盖夜间/逆光/雨雾/长尾类别**。
- **帕累托前沿**：同延迟比 AP，不是同 AP 比延迟。**前沿是口径的函数**——只把 NMS 从
  1.4 ms 改成 0.5 ms，前沿成员就从 8 个变成 10 个。
- **p50 与 p99 会给出相反的结论**：YOLO 式 p50 更快但 p99 更慢。
  top-k 封顶能把 p99 压下来，代价是约 1/3 的帧被截断（截掉的正是远处小目标）。
  另外「无 NMS」也不等于延迟恒定——**跟踪与关联同样依赖目标数**。
- **选型三步**：硬约束过滤 → 用**自己数据上的分桶指标**画前沿 → 加权打分 + 敏感性分析。
  顺序不能反。**用 COCO AP 排序 + 用论文 FPS 排序，两个错误叠加能得出与实测完全相反的结论。**

本课到此结束。下一站：**C54（DETR 集合预测的完整推导）** 与 **C57（小目标）** ——
它们分别补上本课里两个被反复引用但没有展开的东西：匈牙利匹配，和「为什么小目标这么难」。